# Harmony Parameter Sensitivity Analysis

Sweeps Harmony's `theta`, `sigma`, and `nclust` parameters for two-brain
protein-intensity integration and scores each run against an RNA-derived
biology reference.

**Pipeline highlights:**
1. **Hungarian cluster matching** (`scipy.optimize.linear_sum_assignment`)
   gives a globally optimal 1-to-1 pairing that maximizes total Spearman
   correlation across matched pairs.
2. **Small-cluster filter** (< 50 cells per independent Leiden cluster)
   drops noisy clusters before the cross-brain correlation step.
3. **Rebuild + re-PCA** on the filtered joint matrix before the sweep, so
   the PCA Harmony sees isn't tilted by the dropped fringe cells.
4. **`obs_names` uniqueness check** after concat — the downstream
   label-by-name lookups assume unique names and silent collisions would
   misassign reference labels.
5. **Cell-count reports** at three stages: per-brain pre-filter,
   per-brain post-filter, per matched type after Hungarian.


In [ ]:
import anndata as ad
import os
import numpy as np
import scanpy as sc
import harmonypy as hm
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import silhouette_score, adjusted_rand_score, normalized_mutual_info_score
from sklearn.neighbors import NearestNeighbors, kneighbors_graph
from scipy.sparse.csgraph import connected_components
from scipy.stats import spearmanr, chisquare
from scipy.optimize import linear_sum_assignment
from itertools import product


## 1. Load and preprocess

In [ ]:
# Load nuclear protein intensity matrices of the two brain sections.
# Brain 1 (~178927 cells, 18 markers) and Brain 2 (~194890 cells, 15 markers)
# are pre-saved h5ad snapshots produced by the upstream segmentation pipeline.
adata_1st_brain = ad.read_h5ad('input/adata_18_nuclear_46_prot_brain1.h5ad')
adata_2nd_brain = ad.read_h5ad('input/brain2_nuclear_int_18prot.h5ad')

# Prefix obs_names so we can disambiguate same-numbered cells across brains
# AFTER concat (this prevents the row-order alignment bugs we hit earlier).
adata_1st_brain.obs_names = 'b1_' + adata_1st_brain.obs_names.astype(str).str.strip()
adata_2nd_brain.obs_names = 'b2_' + adata_2nd_brain.obs_names.astype(str).str.strip()

common_vars = adata_1st_brain.var_names.intersection(adata_2nd_brain.var_names)
adata_1 = adata_1st_brain[:, common_vars].copy()
adata_2 = adata_2nd_brain[:, common_vars].copy()

In [ ]:
# Load RNA transcript data for both brains.
# Prefix each CSV index with 'b1_' / 'b2_' so they align with the prefixed
# obs_names on adata_b1 / adata_b2.

rna_b1_df = pd.read_csv("input/cell_by_transcript_gene_name_matrix1.csv", index_col=0)
rna_b1_df.index = 'b1_' + rna_b1_df.index.astype(str).str.strip()

rna_b2_df = pd.read_csv("input/cell_by_transcript_gene_name_matrix2.csv", index_col=0)
rna_b2_df.index = 'b2_' + rna_b2_df.index.astype(str).str.strip()

In [ ]:
# Concatenate both brains
adata_both = ad.concat([adata_1, adata_2],
                       join="outer",
                       label="dataset",
                       keys=["1st_brain", "2nd_brain"])

# Uniqueness check: downstream label-by-name lookups (matched_ref assembly,
# small-cluster filter, rebuild) assume every obs_name is unique across the
# joint set. Silent collisions would silently misassign ARI/NMI labels.
if not adata_both.obs_names.is_unique:
    n_dupes = int(adata_both.obs_names.duplicated().sum())
    raise ValueError(
        f"adata_both has {n_dupes} duplicated obs_names across the two brains. "
        f"Disambiguate in the source files (e.g. prefix 1st-brain and 2nd-brain "
        f"cell IDs with their brain tag) and re-run from the load step."
    )

adata_both.layers["counts"] = adata_both.X.copy()
sc.pp.normalize_total(adata_both, inplace=True)
sc.pp.log1p(adata_both)
sc.tl.pca(adata_both, svd_solver="arpack")

print(adata_both)


## 2. Independent clustering per brain

Cluster each brain separately using PCA + neighbors + Leiden, with no
cross-brain information. These labels reflect each brain's own biology
without batch contamination.


In [ ]:
# Split by dataset
adata_b1 = adata_both[adata_both.obs['dataset'] == '1st_brain'].copy()
adata_b2 = adata_both[adata_both.obs['dataset'] == '2nd_brain'].copy()

# Cluster brain 1 independently
sc.tl.pca(adata_b1, svd_solver="arpack")
sc.pp.neighbors(adata_b1, n_neighbors=10, random_state=40, use_rep='X_pca')
sc.tl.leiden(adata_b1, resolution=0.8, random_state=40, key_added='leiden_indep')
print(f"Brain 1: {adata_b1.obs['leiden_indep'].nunique()} independent clusters")

# Cluster brain 2 independently
sc.tl.pca(adata_b2, svd_solver="arpack")
sc.pp.neighbors(adata_b2, n_neighbors=10, random_state=40, use_rep='X_pca')
sc.tl.leiden(adata_b2, resolution=0.8, random_state=40, key_added='leiden_indep')
print(f"Brain 2: {adata_b2.obs['leiden_indep'].nunique()} independent clusters")


In [ ]:
adata_b1.write_h5ad("output/adata_b1_indep_clustered.h5ad")
adata_b2.write_h5ad("output/adata_b2_indep_clustered.h5ad")


In [ ]:
adata_b1 = sc.read_h5ad("output/adata_b1_indep_clustered.h5ad")
adata_b2 = sc.read_h5ad("output/adata_b2_indep_clustered.h5ad")


## 2.5 Inspect cluster sizes & drop small clusters

In [ ]:
# Cell counts per independent cluster, per brain (pre-filter).
# Cluster IDs are independent between brains — '3' in Brain 1 and '3' in
# Brain 2 are not the same thing at this stage.

counts_b1 = adata_b1.obs['leiden_indep'].value_counts().sort_index()
counts_b2 = adata_b2.obs['leiden_indep'].value_counts().sort_index()

counts_df = pd.concat([counts_b1.rename('1st_brain'),
                       counts_b2.rename('2nd_brain')], axis=1).fillna(0).astype(int)
counts_df.index.name = 'leiden_indep (per-brain, not matched)'
counts_df.loc['Total'] = counts_df.sum()

print("Independent cluster sizes per brain (pre-filter):")
print(counts_df.to_string())

# Side-by-side bar plots — one axis per brain so independent labels
# are not visually conflated.
fig, axes = plt.subplots(1, 2, figsize=(16, 5), sharey=False)
for ax, counts, title, color in [
    (axes[0], counts_b1, f'Brain 1 -- {len(counts_b1)} indep clusters', 'tab:blue'),
    (axes[1], counts_b2, f'Brain 2 -- {len(counts_b2)} indep clusters', 'tab:orange'),
]:
    ax.bar(counts.index.astype(str), counts.values, color=color)
    ax.set_title(title)
    ax.set_xlabel('leiden_indep cluster ID')
    ax.set_ylabel('Cell count')
    ax.tick_params(axis='x', rotation=45)
    for x, y in zip(counts.index.astype(str), counts.values):
        ax.text(x, y, f'{y}', ha='center', va='bottom', fontsize=8)
plt.suptitle('Per-brain independent cluster sizes (before filter)', fontsize=14, y=1.02)
plt.tight_layout()
plt.savefig('output/v3_cell_counts_prefilter.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Remove per-brain Leiden clusters with fewer than MIN_CELLS cells.
# Small clusters give noisy mean-expression profiles and tend to produce
# unreliable cross-brain matches, so we drop them before the matching step.
# We also drop those cells from adata_both so downstream matched_ref
# assembly and the Harmony sweep stay consistent.

MIN_CELLS = 50

def _small_clusters(ad_brain, key='leiden_indep', min_cells=MIN_CELLS):
    sizes = ad_brain.obs[key].value_counts()
    return sizes[sizes < min_cells].index.tolist(), sizes

small_b1, sizes_b1 = _small_clusters(adata_b1)
small_b2, sizes_b2 = _small_clusters(adata_b2)

print(f"Brain 1: dropping {len(small_b1)} / {len(sizes_b1)} clusters with < {MIN_CELLS} cells")
if small_b1:
    print("  " + ", ".join(f"{c}({sizes_b1[c]})" for c in small_b1))
print(f"Brain 2: dropping {len(small_b2)} / {len(sizes_b2)} clusters with < {MIN_CELLS} cells")
if small_b2:
    print("  " + ", ".join(f"{c}({sizes_b2[c]})" for c in small_b2))

keep_b1 = ~adata_b1.obs['leiden_indep'].isin(small_b1)
keep_b2 = ~adata_b2.obs['leiden_indep'].isin(small_b2)

n_before_b1, n_before_b2 = adata_b1.n_obs, adata_b2.n_obs
n_before_both = adata_both.n_obs

adata_b1 = adata_b1[keep_b1].copy()
adata_b2 = adata_b2[keep_b2].copy()

# Drop the same cells from adata_both so the sweep / matched_ref stay consistent
keep_names = adata_b1.obs_names.union(adata_b2.obs_names)
adata_both = adata_both[adata_both.obs_names.isin(keep_names)].copy()

# Remove dropped clusters from the categorical index
adata_b1.obs['leiden_indep'] = adata_b1.obs['leiden_indep'].cat.remove_unused_categories()
adata_b2.obs['leiden_indep'] = adata_b2.obs['leiden_indep'].cat.remove_unused_categories()

print()
print(f"Brain 1:    {n_before_b1:>7} -> {adata_b1.n_obs:>7} cells "
      f"({len(adata_b1.obs['leiden_indep'].cat.categories)} clusters remain)")
print(f"Brain 2:    {n_before_b2:>7} -> {adata_b2.n_obs:>7} cells "
      f"({len(adata_b2.obs['leiden_indep'].cat.categories)} clusters remain)")
print(f"adata_both: {n_before_both:>7} -> {adata_both.n_obs:>7} cells")


## 3. Match clusters across brains using RNA expression

Compute the mean RNA expression profile (79 genes) for each cluster in each
brain, then correlate brain-1 clusters against brain-2 clusters. Using RNA
instead of the 15-protein matrix gives much better discrimination — protein
correlations are uniformly high and not informative for matching.


In [ ]:
# Common genes across both brains
common_genes = rna_b1_df.columns.intersection(rna_b2_df.columns)
rna_b1_df = rna_b1_df[common_genes]
rna_b2_df = rna_b2_df[common_genes]

# Match to cells in the FILTERED adata_b1 / adata_b2
common_b1 = adata_b1.obs_names.intersection(rna_b1_df.index)
common_b2 = adata_b2.obs_names.intersection(rna_b2_df.index)
rna_b1_df = rna_b1_df.loc[common_b1]
rna_b2_df = rna_b2_df.loc[common_b2]

# Normalize RNA: total-count normalize + log1p per brain
rna_b1 = ad.AnnData(X=rna_b1_df.to_numpy(), obs=adata_b1[common_b1].obs.copy(),
                     var=pd.DataFrame(index=common_genes))
sc.pp.normalize_total(rna_b1, inplace=True)
sc.pp.log1p(rna_b1)

rna_b2 = ad.AnnData(X=rna_b2_df.to_numpy(), obs=adata_b2[common_b2].obs.copy(),
                     var=pd.DataFrame(index=common_genes))
sc.pp.normalize_total(rna_b2, inplace=True)
sc.pp.log1p(rna_b2)

print(f"RNA: {len(common_genes)} common genes")
print(f"Brain 1: {rna_b1.n_obs} cells matched, Brain 2: {rna_b2.n_obs} cells matched")


In [ ]:
# UMAP per brain on the filtered set
import umap

reducer_b1 = umap.UMAP(n_neighbors=5, min_dist=0, n_components=2, random_state=42)
adata_b1.obsm['X_umap'] = reducer_b1.fit_transform(adata_b1.obsm['X_pca'])

reducer_b2 = umap.UMAP(n_neighbors=5, min_dist=0, n_components=2, random_state=42)
adata_b2.obsm['X_umap'] = reducer_b2.fit_transform(adata_b2.obsm['X_pca'])

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
for ax, ad_brain, title in [(axes[0], adata_b1, 'Brain 1'), (axes[1], adata_b2, 'Brain 2')]:
    labels = ad_brain.obs['leiden_indep'].astype('category')
    cats = labels.cat.categories
    n_cl = len(cats)
    cmap = plt.cm.get_cmap('tab20', max(n_cl, 20))
    coords = ad_brain.obsm['X_umap']
    for j, cat in enumerate(cats):
        mask = labels == cat
        ax.scatter(coords[mask, 0], coords[mask, 1],
                   s=1, c=[cmap(j)], alpha=0.4, rasterized=True)
        cx, cy = coords[mask, 0].mean(), coords[mask, 1].mean()
        ax.text(cx, cy, str(cat), fontsize=8, fontweight='bold', ha='center', va='center',
                bbox=dict(boxstyle='round,pad=0.15', fc='white', alpha=0.7, lw=0))
    ax.set_title(f'{title} -- independent Leiden ({n_cl} clusters, post-filter)', fontsize=14)
    ax.set_xlabel('UMAP1'); ax.set_ylabel('UMAP2')
plt.suptitle('Independent clustering per brain (UMAP, post-filter)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('output/v3_indep_umap.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Spatial XY positions colored by independent cluster (post-filter)
spatial_b1 = adata_1st_brain.obsm['spatial']
spatial_b1_matched = spatial_b1[adata_1st_brain.obs_names.get_indexer(adata_b1.obs_names)]

spatial_b2 = adata_2nd_brain.obsm['spatial']
spatial_b2_matched = spatial_b2[adata_2nd_brain.obs_names.get_indexer(adata_b2.obs_names)]

fig, axes = plt.subplots(1, 2, figsize=(24, 10))
for ax, ad_brain, xy, title in [
    (axes[0], adata_b1, spatial_b1_matched, 'Brain 1'),
    (axes[1], adata_b2, spatial_b2_matched, 'Brain 2')
]:
    labels = ad_brain.obs['leiden_indep'].astype('category')
    cats = labels.cat.categories
    n_cl = len(cats)
    cmap = plt.cm.get_cmap('tab20', max(n_cl, 20))
    for j, cat in enumerate(cats):
        mask = labels == cat
        ax.scatter(xy[mask, 0], xy[mask, 1],
                   s=1, c=[cmap(j)], alpha=0.4, rasterized=True)
    ax.set_title(f'{title} -- spatial ({n_cl} clusters, post-filter)', fontsize=14)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_aspect('equal'); ax.invert_yaxis()
plt.suptitle('Independent clustering per brain (spatial XY, post-filter)', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('output/v3_indep_spatial.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Top marker genes per cluster in each brain
sc.tl.rank_genes_groups(rna_b1, groupby="leiden_indep", method="wilcoxon", n_genes=4)
sc.tl.rank_genes_groups(rna_b2, groupby="leiden_indep", method="wilcoxon", n_genes=4)

top_genes_b1 = set()
for cl in rna_b1.obs["leiden_indep"].unique():
    top_genes_b1.update(sc.get.rank_genes_groups_df(rna_b1, group=cl).head(4)["names"].tolist())

top_genes_b2 = set()
for cl in rna_b2.obs["leiden_indep"].unique():
    top_genes_b2.update(sc.get.rank_genes_groups_df(rna_b2, group=cl).head(4)["names"].tolist())

top_genes = sorted(top_genes_b1 | top_genes_b2)
print(f"Using {len(top_genes)} marker genes (union of top 4 per cluster from both brains)")
print(top_genes)

# Mean RNA expression per cluster using only top marker genes
b1_clusters = sorted(adata_b1.obs["leiden_indep"].unique(), key=int)
b2_clusters = sorted(adata_b2.obs["leiden_indep"].unique(), key=int)

gene_idx_b1 = [list(rna_b1.var_names).index(g) for g in top_genes]
gene_idx_b2 = [list(rna_b2.var_names).index(g) for g in top_genes]

b1_means = np.array([
    rna_b1[rna_b1.obs["leiden_indep"] == c].X[:, gene_idx_b1].mean(axis=0)
    for c in b1_clusters
])
if b1_means.ndim > 2:
    b1_means = b1_means.reshape(len(b1_clusters), -1)

b2_means = np.array([
    rna_b2[rna_b2.obs["leiden_indep"] == c].X[:, gene_idx_b2].mean(axis=0)
    for c in b2_clusters
])
if b2_means.ndim > 2:
    b2_means = b2_means.reshape(len(b2_clusters), -1)

# Spearman correlation matrix
corr_matrix = np.zeros((len(b1_clusters), len(b2_clusters)))
for i in range(len(b1_clusters)):
    for j in range(len(b2_clusters)):
        corr_matrix[i, j] = spearmanr(b1_means[i], b2_means[j]).correlation

corr_df = pd.DataFrame(corr_matrix,
                        index=[f"B1_{c}" for c in b1_clusters],
                        columns=[f"B2_{c}" for c in b2_clusters])

print(f"Cross-brain cluster correlation matrix ({len(top_genes)} marker genes):")
print(corr_df.round(3))


In [ ]:
# Heatmap of cross-brain cluster correlations
fig, ax = plt.subplots(figsize=(12, 8))
sns.heatmap(corr_df, annot=True, fmt='.2f', cmap='RdYlBu_r', vmin=-1, vmax=1,
            square=True, ax=ax, linewidths=0)
ax.set_title('Spearman correlation: Brain 1 vs Brain 2 independent clusters', fontsize=14)
ax.set_xlabel('Brain 2 clusters'); ax.set_ylabel('Brain 1 clusters')
plt.tight_layout()
plt.savefig('output/v3_cross_brain_correlation.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# Hungarian matching: globally optimal 1-to-1 pairing.
# linear_sum_assignment minimizes total cost, so negate corr_matrix
# to maximize total Spearman correlation across matched pairs.

cost = -corr_matrix
row_ind, col_ind = linear_sum_assignment(cost)

matched_b1 = {}
matched_b2 = {}
used_b1 = set()
used_b2 = set()
type_id = 0

for i, j in zip(row_ind, col_ind):
    corr_val = corr_matrix[i, j]
    c1 = b1_clusters[i]
    c2 = b2_clusters[j]
    if corr_val < 0.5:                   # minimum correlation threshold
        continue
    label = f'type_{type_id}'
    matched_b1[c1] = label
    matched_b2[c2] = label
    used_b1.add(c1)
    used_b2.add(c2)
    print(f'{label}: B1 cluster {c1} <-> B2 cluster {c2}  (rho={corr_val:.3f})')
    type_id += 1

for c1 in b1_clusters:
    if c1 not in matched_b1:
        matched_b1[c1] = f'b1_only_{c1}'
        print(f'b1_only_{c1}: B1 cluster {c1} (no match)')

for c2 in b2_clusters:
    if c2 not in matched_b2:
        matched_b2[c2] = f'b2_only_{c2}'
        print(f'b2_only_{c2}: B2 cluster {c2} (no match)')

print()
print(f'Matched types: {type_id}, Unmatched B1: {len(b1_clusters)-len(used_b1)}, '
      f'Unmatched B2: {len(b2_clusters)-len(used_b2)}')


In [ ]:
# Spatial scatter plots with matched clusters sharing the same color
all_types = sorted(set(matched_b1.values()) | set(matched_b2.values()))
cmap = plt.cm.get_cmap('tab20', max(len(all_types), 20))
type_colors = {t: cmap(i) for i, t in enumerate(all_types)}

b1_types = adata_b1.obs['leiden_indep'].map(matched_b1)
b2_types = adata_b2.obs['leiden_indep'].map(matched_b2)

fig, axes = plt.subplots(1, 2, figsize=(24, 14))
for ax, types, xy, title in [
    (axes[0], b1_types, spatial_b1_matched, 'Brain 1'),
    (axes[1], b2_types, spatial_b2_matched, 'Brain 2')
]:
    for t in all_types:
        mask = (types == t).values
        if mask.sum() == 0:
            continue
        label_short = t.replace('type_', 'T')
        ax.scatter(xy[mask, 0], xy[mask, 1],
                   s=1, c=[type_colors[t]], alpha=0.4, rasterized=True, label=label_short)
    ax.set_title(f'{title} -- matched clusters', fontsize=14)
    ax.set_xlabel('X'); ax.set_ylabel('Y')
    ax.set_aspect('equal'); ax.invert_yaxis()

handles, labels = axes[0].get_legend_handles_labels()
fig.legend(handles, labels, loc='center right', markerscale=5, fontsize=18,
           bbox_to_anchor=(1.08, 0.5), title='Matched type')
plt.suptitle('Spatial distribution -- corresponding clusters share colors', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig('output/v3_matched_clusters_spatial.pdf', dpi=200, bbox_inches='tight')
plt.show()


## 3.5 Save per-brain h5ad with matched cluster labels

Each cell now has both its independent Leiden cluster (`leiden_indep`) and its
Hungarian-matched cross-brain type (`matched_type`). Cells whose independent
cluster did not pair with any cluster in the other brain keep a `b1_only_*` /
`b2_only_*` label so they stay identifiable downstream. The matched-pair table
is stored in `.uns['matched_type_pairs']` so the file is self-describing.


In [ ]:
# Attach matched_type and a matched/unmatched status flag to each per-brain AnnData.
b1_matched_type = adata_b1.obs['leiden_indep'].map(matched_b1).astype(str)
b2_matched_type = adata_b2.obs['leiden_indep'].map(matched_b2).astype(str)

adata_b1.obs['matched_type'] = pd.Categorical(b1_matched_type)
adata_b2.obs['matched_type'] = pd.Categorical(b2_matched_type)

adata_b1.obs['matched_status'] = pd.Categorical(
    np.where(b1_matched_type.str.startswith('type_'), 'matched', 'unmatched'),
    categories=['matched', 'unmatched'],
)
adata_b2.obs['matched_status'] = pd.Categorical(
    np.where(b2_matched_type.str.startswith('type_'), 'matched', 'unmatched'),
    categories=['matched', 'unmatched'],
)

# Pairing table: matched_type, brain-1 cluster, brain-2 cluster, Spearman rho.
pair_rows = []
b2_label_to_cluster = {label: c for c, label in matched_b2.items()}
for c1, label in matched_b1.items():
    if not label.startswith('type_'):
        continue
    c2 = b2_label_to_cluster.get(label)
    if c2 is None:
        continue
    rho = corr_matrix[b1_clusters.index(c1), b2_clusters.index(c2)]
    pair_rows.append({
        'matched_type': label,
        'b1_cluster': c1,
        'b2_cluster': c2,
        'spearman_rho': float(rho),
    })

pair_df = pd.DataFrame(pair_rows)
if not pair_df.empty:
    pair_df['_ord'] = pair_df['matched_type'].str.replace('type_', '', regex=False).astype(int)
    pair_df = pair_df.sort_values('_ord').drop(columns='_ord').reset_index(drop=True)

adata_b1.uns['matched_type_pairs'] = pair_df
adata_b2.uns['matched_type_pairs'] = pair_df

adata_b1.write_h5ad('output/adata_b1_matched_clusters.h5ad')
adata_b2.write_h5ad('output/adata_b2_matched_clusters.h5ad')

# Also export the pairing table as a flat CSV for spreadsheet inspection.
pair_df.to_csv('output/matched_cluster_pairs.csv', index=False)

print('Wrote:')
print('  output/adata_b1_matched_clusters.h5ad')
print('  output/adata_b2_matched_clusters.h5ad')
print('  output/matched_cluster_pairs.csv')
print()
print(f"Brain 1: {adata_b1.n_obs:>6} cells "
      f"({(adata_b1.obs['matched_status']=='matched').sum():>6} matched, "
      f"{(adata_b1.obs['matched_status']=='unmatched').sum():>6} unmatched)")
print(f"Brain 2: {adata_b2.n_obs:>6} cells "
      f"({(adata_b2.obs['matched_status']=='matched').sum():>6} matched, "
      f"{(adata_b2.obs['matched_status']=='unmatched').sum():>6} unmatched)")
print()
print('Pairing table (also in .uns["matched_type_pairs"]):')
print(pair_df.to_string(index=False))


In [ ]:
# Build reference labels for every cell in adata_both
ref_labels = pd.Series(index=adata_both.obs_names, dtype=str)

for c1, label in matched_b1.items():
    mask = (adata_b1.obs['leiden_indep'] == c1)
    ref_labels.loc[adata_b1.obs_names[mask]] = label

for c2, label in matched_b2.items():
    mask = (adata_b2.obs['leiden_indep'] == c2)
    ref_labels.loc[adata_b2.obs_names[mask]] = label

adata_both.obs['matched_ref'] = ref_labels.values

print("Reference label distribution:")
print(adata_both.obs['matched_ref'].value_counts())


In [ ]:
# Cell counts per matched type x brain (after Hungarian matching)
counts = pd.crosstab(adata_both.obs['matched_ref'],
                     adata_both.obs['dataset'],
                     margins=True, margins_name='Total')

def _sort_key(label):
    if label == 'Total':             return (3, 0, '')
    if label.startswith('type_'):    return (0, int(label.split('_')[1]), '')
    if label.startswith('b1_only_'): return (1, 0, label)
    if label.startswith('b2_only_'): return (2, 0, label)
    return (4, 0, label)

counts = counts.loc[sorted(counts.index, key=_sort_key)]
print("Cell counts per matched type:")
print(counts.to_string())

plot_df = counts.drop(index='Total').drop(columns='Total')
fig, ax = plt.subplots(figsize=(max(8, 0.4 * len(plot_df)), 5))
plot_df.plot(kind='bar', ax=ax, width=0.8,
             color={'1st_brain': 'tab:blue', '2nd_brain': 'tab:orange'})
ax.set_xlabel('Matched type')
ax.set_ylabel('Cell count')
ax.set_title('Cells per matched type, by brain')
ax.legend(title='Dataset')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.savefig('output/v3_cell_counts_matched.pdf', dpi=200, bbox_inches='tight')
plt.show()


## 4. Rebuild joint matrix on the filtered set

The PCA stored in `adata_both.obsm['X_pca']` was computed in the concat cell
on the pre-filter joint set. We rebuild `adata_both` from the filtered
per-brain AnnDatas and re-run PCA so the Harmony sweep operates on a
representation that isn't tilted by dropped fringe cells.


In [ ]:
# Rebuild adata_both from the filtered per-brain AnnDatas and re-PCA.
# matched_ref has to be reattached after the rebuild since ad.concat resets obs.

_matched_ref_map = dict(zip(adata_both.obs_names, adata_both.obs['matched_ref']))

adata_both = ad.concat(
    [adata_b1, adata_b2],
    join="outer",
    label="dataset",
    keys=["1st_brain", "2nd_brain"],
)

# adata_b1.X / adata_b2.X are already log1p-normalized (inherited from the
# concat cell, before the split). Per-cell normalize_total scale factors are
# independent of which other cells are in the matrix, so re-normalizing would
# be a no-op for retained cells. We just need a fresh PCA on the filtered set.
sc.tl.pca(adata_both, svd_solver="arpack")

adata_both.obs['matched_ref'] = adata_both.obs_names.map(_matched_ref_map).astype(str)

assert not adata_both.obs['matched_ref'].isin(['nan', 'None']).any(), \
    "Some cells lost their matched_ref label during rebuild -- check obs_names alignment"

print(f"Rebuilt adata_both: {adata_both.n_obs} cells x {adata_both.n_vars} features")
print(f"Fresh PCA shape:    {adata_both.obsm['X_pca'].shape}")
print(f"Batches:            {adata_both.obs['dataset'].value_counts().to_dict()}")
print(f"Matched types:      {adata_both.obs['matched_ref'].nunique()} unique labels")


## 5. Parameter sweep

ARI/NMI are computed against the Hungarian-matched independent cluster
labels -- a biology reference that is independent of Harmony's input
(protein PCA) and not contaminated by batch effects.


### 5a. Helper functions for the supplementary metrics

The four standard global scores (`sil_batch`, `sil_cluster`, ARI, NMI) can hide
two failure modes that are visible by eye in the integrated UMAP:

* **batches not mixing locally** -- a global silhouette near 0 can come from a
  small fraction of well-mixed cells, while large regions stay single-brain.
* **matched types not co-clustering across brains** -- ARI/NMI vs `matched_ref`
  reward Leiden agreement with the type label, but do not require Brain 1 and
  Brain 2 cells of the same matched type to land in the same Leiden cluster.

The helpers below add four local batch-mixing metrics and four cross-brain
correspondence metrics so each sweep row is scored on both axes.


In [ ]:
# kNN-based metrics share one NN search per sweep iteration to keep cost down.
KNN_K = 30


def _knn_indices(X, k=KNN_K):
    nn = NearestNeighbors(n_neighbors=k + 1).fit(X)
    _, ind = nn.kneighbors(X)
    return ind[:, 1:]  # drop self


# ---- Local batch mixing -----------------------------------------------------

def ilisi_per_type(X, batch, types, knn_ind=None, k=KNN_K):
    """Inverse Simpson diversity of `batch` in each cell's kNN, averaged within
    each matched type then across types. Range ~1 (single brain) to n_batches
    (perfectly mixed). Higher = better local mixing inside each type."""
    if knn_ind is None:
        knn_ind = _knn_indices(X, k=k)
    batch_arr = np.asarray(batch)
    type_arr = np.asarray(types)
    nb_batch = batch_arr[knn_ind]
    cats = np.unique(batch_arr)
    counts = np.column_stack([(nb_batch == c).sum(axis=1) for c in cats]).astype(float)
    p = counts / counts.sum(axis=1, keepdims=True)
    lisi = 1.0 / (p ** 2).sum(axis=1)
    means = [lisi[type_arr == t].mean() for t in np.unique(type_arr) if (type_arr == t).any()]
    return float(np.mean(means)) if means else np.nan


def kbet_acceptance(X, batch, knn_ind=None, k=KNN_K, n_test=2000, seed=42):
    """Fraction of test cells whose kNN batch composition is consistent with
    the global batch frequency (chi-square p > 0.05). Higher = better mixing."""
    if knn_ind is None:
        knn_ind = _knn_indices(X, k=k)
    batch_arr = np.asarray(batch)
    cats, glob = np.unique(batch_arr, return_counts=True)
    expected_freq = glob / glob.sum()
    rng = np.random.default_rng(seed)
    test_idx = rng.choice(len(X), size=min(n_test, len(X)), replace=False)
    accepted = 0
    for i in test_idx:
        nb = batch_arr[knn_ind[i]]
        obs = np.array([(nb == c).sum() for c in cats], dtype=float)
        exp = expected_freq * obs.sum()
        if (exp > 0).all():
            _, p = chisquare(obs, exp)
            if p > 0.05:
                accepted += 1
    return accepted / len(test_idx)


def graph_connectivity_per_type(X, types, k=15):
    """For each matched type, build a kNN graph on its cells and report the
    largest-connected-component fraction, averaged across types. 1.0 = each
    type forms a single blob; ~0.5 = each type splits into two batch islands."""
    type_arr = np.asarray(types)
    scores = []
    for t in np.unique(type_arr):
        m = type_arr == t
        n = int(m.sum())
        if n < 5:
            continue
        kk = min(k, n - 1)
        if kk < 1:
            continue
        g = kneighbors_graph(X[m], kk, mode='connectivity', include_self=False)
        _, lbl = connected_components(g, directed=False)
        _, sizes = np.unique(lbl, return_counts=True)
        scores.append(sizes.max() / n)
    return float(np.mean(scores)) if scores else np.nan


def batch_asw_per_type(X, batch, types):
    """Batch silhouette computed *within* each matched type then averaged.
    Reported as 1 - |asw|, so higher = better batch mixing inside the type."""
    type_arr = np.asarray(types)
    batch_arr = np.asarray(batch)
    scores = []
    for t in np.unique(type_arr):
        m = type_arr == t
        if m.sum() < 10:
            continue
        if len(np.unique(batch_arr[m])) < 2:
            continue
        try:
            s = silhouette_score(X[m], batch_arr[m])
            scores.append(1 - abs(s))
        except Exception:
            continue
    return float(np.mean(scores)) if scores else np.nan


# ---- Cross-brain cluster correspondence -------------------------------------

def co_cluster_purity(post, types, batch):
    """Cell-weighted fraction of matched types whose dominant post-Harmony
    cluster among Brain 1 cells equals the dominant cluster among Brain 2
    cells. 1.0 = every matched type co-clusters across brains."""
    type_arr = np.asarray(types)
    batch_arr = np.asarray(batch)
    post_arr = np.asarray(post)
    cats = np.unique(batch_arr)
    if len(cats) != 2:
        return np.nan
    b1, b2 = cats
    matched_cells = 0
    total_cells = 0
    for t in np.unique(type_arr):
        if not str(t).startswith('type_'):
            continue
        m1 = (type_arr == t) & (batch_arr == b1)
        m2 = (type_arr == t) & (batch_arr == b2)
        if m1.sum() == 0 or m2.sum() == 0:
            continue
        c1 = pd.Series(post_arr[m1]).mode().iloc[0]
        c2 = pd.Series(post_arr[m2]).mode().iloc[0]
        n = int(m1.sum() + m2.sum())
        total_cells += n
        if c1 == c2:
            matched_cells += n
    return matched_cells / total_cells if total_cells else np.nan


def within_type_batch_ari(post, types, batch):
    """For each matched type, ARI between batch label and post-Harmony cluster
    label, averaged across types. Lower (toward 0) = clusters are independent
    of brain within the type, i.e. brains co-cluster."""
    type_arr = np.asarray(types)
    batch_arr = np.asarray(batch)
    post_arr = np.asarray(post)
    aris = []
    for t in np.unique(type_arr):
        if not str(t).startswith('type_'):
            continue
        m = type_arr == t
        if m.sum() < 10:
            continue
        if len(np.unique(batch_arr[m])) < 2:
            continue
        if len(np.unique(post_arr[m])) < 2:
            aris.append(0.0)
            continue
        aris.append(adjusted_rand_score(batch_arr[m], post_arr[m]))
    return float(np.mean(aris)) if aris else np.nan


def centroid_dist_per_type(X, types, batch):
    """Per matched type, L2 distance between Brain 1 and Brain 2 centroids in
    the embedding, normalized by the within-type stddev norm and averaged
    across types. Lower = matched-type centroids align better across brains."""
    type_arr = np.asarray(types)
    batch_arr = np.asarray(batch)
    cats = np.unique(batch_arr)
    if len(cats) != 2:
        return np.nan
    b1, b2 = cats
    dists = []
    for t in np.unique(type_arr):
        if not str(t).startswith('type_'):
            continue
        m1 = (type_arr == t) & (batch_arr == b1)
        m2 = (type_arr == t) & (batch_arr == b2)
        if m1.sum() < 5 or m2.sum() < 5:
            continue
        c1 = X[m1].mean(axis=0)
        c2 = X[m2].mean(axis=0)
        d = float(np.linalg.norm(c1 - c2))
        spread = float(np.linalg.norm(X[type_arr == t].std(axis=0)))
        if spread > 0:
            dists.append(d / spread)
    return float(np.mean(dists)) if dists else np.nan


def mnn_hit_rate(X, batch, types, knn_ind=None, k=KNN_K):
    """For each cell, fraction of its kNN that are (other-brain) AND (same
    matched type), averaged over cells. Penalizes both poor batch mixing and
    biology smear -- a cell needs cross-brain neighbors of the right type."""
    if knn_ind is None:
        knn_ind = _knn_indices(X, k=k)
    batch_arr = np.asarray(batch)
    type_arr = np.asarray(types)
    cats = np.unique(batch_arr)
    if len(cats) != 2:
        return np.nan
    b1, b2 = cats
    other = np.where(batch_arr == b1, b2, b1)
    nb_batch = batch_arr[knn_ind]
    nb_type = type_arr[knn_ind]
    other_batch = nb_batch == other[:, None]
    same_type = nb_type == type_arr[:, None]
    hits = (same_type & other_batch).sum(axis=1)
    return float(hits.mean() / knn_ind.shape[1])


def hungarian_recovery_rate(post, batch, ref, cell_names,
                             rna_b1_ad, rna_b2_ad, marker_genes,
                             original_types, threshold=0.5):
    """Re-run the section-3 RNA Hungarian matching on per-brain `leiden_harmony`
    clusters and return the fraction of the original `type_X` pairings that are
    reproduced.

    For each post-Harmony Leiden cluster split by brain, compute the mean
    expression on `marker_genes`, build the Spearman correlation matrix between
    brain-1 and brain-2 per-brain clusters, and run linear_sum_assignment.
    A new pair (h1, h2) reproduces `type_X` if (i) the dominant matched_ref
    label among h1's brain-1 cells equals `type_X`, (ii) the dominant label
    among h2's brain-2 cells equals `type_X`, and (iii) Hungarian links them
    with rho >= `threshold`. Higher = Harmony preserved more of the biology
    that the original matching was based on."""
    from collections import Counter

    cats = np.unique(batch)
    if len(cats) != 2 or not original_types:
        return np.nan
    b1_name, b2_name = cats[0], cats[1]

    try:
        gene_idx_b1 = [rna_b1_ad.var_names.get_loc(g) for g in marker_genes]
        gene_idx_b2 = [rna_b2_ad.var_names.get_loc(g) for g in marker_genes]
    except KeyError:
        return np.nan

    cell_names = np.asarray(cell_names)
    post_arr = np.asarray(post)
    batch_arr = np.asarray(batch)
    ref_arr = np.asarray(ref)

    def _per_brain(mask, rna_ad, gene_idx):
        names = cell_names[mask]
        labs = post_arr[mask]
        types = ref_arr[mask]
        in_rna = np.isin(names, rna_ad.obs_names.values)
        names = names[in_rna]; labs = labs[in_rna]; types = types[in_rna]
        if len(names) == 0:
            return [], np.zeros((0, len(gene_idx))), {}
        X = rna_ad[names].X
        if hasattr(X, 'toarray'):
            X = X.toarray()
        X = np.asarray(X)[:, gene_idx]
        kept_clusters, means, dominant = [], [], {}
        for c in sorted(set(labs)):
            m = labs == c
            if m.sum() < 5:
                continue
            kept_clusters.append(c)
            means.append(X[m].mean(axis=0))
            tlist = [t for t in types[m] if str(t).startswith('type_')]
            if tlist:
                dominant[c] = Counter(tlist).most_common(1)[0][0]
        means_arr = np.array(means) if means else np.zeros((0, len(gene_idx)))
        return kept_clusters, means_arr, dominant

    h1_clusters, h1_means, h1_dom = _per_brain(batch_arr == b1_name, rna_b1_ad, gene_idx_b1)
    h2_clusters, h2_means, h2_dom = _per_brain(batch_arr == b2_name, rna_b2_ad, gene_idx_b2)

    if len(h1_clusters) == 0 or len(h2_clusters) == 0:
        return np.nan

    corr = np.zeros((len(h1_clusters), len(h2_clusters)))
    for i in range(len(h1_clusters)):
        for j in range(len(h2_clusters)):
            r = spearmanr(h1_means[i], h2_means[j]).correlation
            corr[i, j] = -1.0 if (r is None or np.isnan(r)) else r

    row_ind, col_ind = linear_sum_assignment(-corr)
    recovered = set()
    for i, j in zip(row_ind, col_ind):
        if corr[i, j] < threshold:
            continue
        t1 = h1_dom.get(h1_clusters[i])
        t2 = h2_dom.get(h2_clusters[j])
        if t1 is not None and t1 == t2 and t1 in original_types:
            recovered.add(t1)

    return len(recovered) / len(original_types)


In [ ]:
# Parameter grid. Sigma is fixed at 0.02 -- prior sweeps showed sigma had a
# small, monotone effect compared to theta and nclust, and fixing it here keeps
# the sweep cost down so the supplementary metrics fit in the same runtime.
theta_values = [1, 2, 4, 6, 8, 12]
sigma_values = [0.02]
nclust_values = [30, 50, 100, 150]
lamb_fixed = 0.3

# Subsample for speed
np.random.seed(42)
n_subsample = 50000
if adata_both.n_obs > n_subsample:
    idx = np.random.choice(adata_both.n_obs, n_subsample, replace=False)
    adata_sub = adata_both[idx].copy()
else:
    adata_sub = adata_both.copy()

ref_labels_sub = adata_sub.obs['matched_ref'].values
batch_labels_sub = adata_sub.obs['dataset'].values
X_pca_sub = adata_sub.obsm['X_pca'].copy()

# Needed by hungarian_recovery_rate: cell ids to map into rna_b1 / rna_b2,
# and the set of original type_X labels we want to see reproduced.
cell_names_sub = adata_sub.obs_names.values
original_types_set = {t for t in np.unique(ref_labels_sub) if str(t).startswith('type_')}

print(f"Running sweep on {adata_sub.n_obs} cells")
print(f"Reference labels: {len(np.unique(ref_labels_sub))} unique types "
      f"({len(original_types_set)} matched type_X)")


In [ ]:
results = []

total = len(theta_values) * len(sigma_values) * len(nclust_values)
count = 0

# All-NaN row used when a Harmony fit fails to converge.
_NAN_FIELDS = [
    'sil_batch', 'sil_cluster', 'ari', 'nmi', 'n_clusters',
    'ilisi_per_type', 'kbet_accept', 'graph_conn_per_type', 'batch_asw_per_type',
    'co_cluster_purity', 'within_type_batch_ari',
    'centroid_dist_per_type', 'mnn_hit_rate',
    'pair_recovery_rate',
]

for theta, sigma, nclust in product(theta_values, sigma_values, nclust_values):
    count += 1
    print(f"[{count}/{total}] theta={theta}, sigma={sigma}, nclust={nclust}", end=" ")

    try:
        ho = hm.run_harmony(
            X_pca_sub, adata_sub.obs, 'dataset',
            theta=theta, lamb=lamb_fixed, sigma=sigma,
            nclust=nclust, max_iter_harmony=20, random_state=42
        )
        X_harmony = ho.Z_corr.T

        adata_sub.obsm['X_harmony_test'] = X_harmony
        sc.pp.neighbors(adata_sub, n_neighbors=10, random_state=42, use_rep='X_harmony_test')
        sc.tl.leiden(adata_sub, resolution=0.8, random_state=38, key_added='leiden_harmony')

        post_labels = adata_sub.obs['leiden_harmony'].values

        # Original four global scores
        sil_batch = silhouette_score(X_harmony, batch_labels_sub, sample_size=10000, random_state=42)
        sil_cluster = silhouette_score(X_harmony, post_labels, sample_size=10000, random_state=42)
        ari = adjusted_rand_score(ref_labels_sub, post_labels)
        nmi = normalized_mutual_info_score(ref_labels_sub, post_labels)
        n_clusters = adata_sub.obs['leiden_harmony'].nunique()

        # One kNN search reused across iLISI, kBET, MNN
        knn_ind = _knn_indices(X_harmony, k=KNN_K)

        # Local batch-mixing metrics
        ilisi_t = ilisi_per_type(X_harmony, batch_labels_sub, ref_labels_sub, knn_ind=knn_ind)
        kbet = kbet_acceptance(X_harmony, batch_labels_sub, knn_ind=knn_ind)
        graph_conn = graph_connectivity_per_type(X_harmony, ref_labels_sub)
        batch_asw_t = batch_asw_per_type(X_harmony, batch_labels_sub, ref_labels_sub)

        # Cross-brain correspondence metrics
        co_purity = co_cluster_purity(post_labels, ref_labels_sub, batch_labels_sub)
        wt_batch_ari = within_type_batch_ari(post_labels, ref_labels_sub, batch_labels_sub)
        cent_dist = centroid_dist_per_type(X_harmony, ref_labels_sub, batch_labels_sub)
        mnn_hit = mnn_hit_rate(X_harmony, batch_labels_sub, ref_labels_sub, knn_ind=knn_ind)

        # Re-run RNA Hungarian on per-brain leiden_harmony clusters: direct
        # test of whether Harmony preserved the biology that the original
        # type_X matching was based on.
        pair_recovery = hungarian_recovery_rate(
            post_labels, batch_labels_sub, ref_labels_sub, cell_names_sub,
            rna_b1, rna_b2, top_genes, original_types_set, threshold=0.5,
        )

        results.append({
            'theta': theta, 'sigma': sigma, 'nclust': nclust,
            'sil_batch': sil_batch, 'sil_cluster': sil_cluster,
            'ari': ari, 'nmi': nmi, 'n_clusters': n_clusters,
            'ilisi_per_type': ilisi_t,
            'kbet_accept': kbet,
            'graph_conn_per_type': graph_conn,
            'batch_asw_per_type': batch_asw_t,
            'co_cluster_purity': co_purity,
            'within_type_batch_ari': wt_batch_ari,
            'centroid_dist_per_type': cent_dist,
            'mnn_hit_rate': mnn_hit,
            'pair_recovery_rate': pair_recovery,
            'converged': True,
        })
        print(f"-> silB={sil_batch:.2f} silC={sil_cluster:.2f} "
              f"ARI={ari:.2f} NMI={nmi:.2f} | "
              f"iLISI={ilisi_t:.2f} kBET={kbet:.2f} "
              f"gConn={graph_conn:.2f} bASW={batch_asw_t:.2f} | "
              f"coPur={co_purity:.2f} wtARI={wt_batch_ari:.2f} "
              f"cDist={cent_dist:.2f} MNN={mnn_hit:.2f} "
              f"pRec={pair_recovery:.2f} | k={n_clusters}")

    except Exception as e:
        print(f"-> FAILED: {e}")
        row = {'theta': theta, 'sigma': sigma, 'nclust': nclust, 'converged': False}
        for f in _NAN_FIELDS:
            row[f] = np.nan
        results.append(row)

results_df = pd.DataFrame(results)
results_df.to_csv("output/harmony_sensitivity_v3_results.csv", index=False)
print()
print(f"Done. Saved {len(results_df)} results to output/harmony_sensitivity_v3_results.csv")


## 5.5 Sigma sweep at chosen (theta=6, nclust=100)

The main sweep above fixes sigma at 0.02. To document sigma's marginal effect,
this section varies sigma across roughly an order of magnitude with theta and
nclust pinned at the chosen values. The same thirteen metrics are recorded so
the sigma curves drop straight into the visualization function used for theta
and nclust.


In [ ]:
# Sigma sweep: theta and nclust pinned to the chosen values, sigma varied.
sigma_sweep_values = [0.005, 0.01, 0.02, 0.05, 0.1, 0.2]
sigma_theta = 6
sigma_nclust = 100

sigma_results = []
for s in sigma_sweep_values:
    print(f"sigma={s}", end=" ")
    try:
        ho = hm.run_harmony(
            X_pca_sub, adata_sub.obs, 'dataset',
            theta=sigma_theta, lamb=lamb_fixed, sigma=s,
            nclust=sigma_nclust, max_iter_harmony=20, random_state=42
        )
        X_harmony = ho.Z_corr.T

        adata_sub.obsm['X_harmony_test'] = X_harmony
        sc.pp.neighbors(adata_sub, n_neighbors=10, random_state=42, use_rep='X_harmony_test')
        sc.tl.leiden(adata_sub, resolution=0.8, random_state=38, key_added='leiden_harmony')
        post_labels = adata_sub.obs['leiden_harmony'].values

        # Original four global scores
        sil_batch = silhouette_score(X_harmony, batch_labels_sub, sample_size=10000, random_state=42)
        sil_cluster = silhouette_score(X_harmony, post_labels, sample_size=10000, random_state=42)
        ari = adjusted_rand_score(ref_labels_sub, post_labels)
        nmi = normalized_mutual_info_score(ref_labels_sub, post_labels)
        n_clusters = adata_sub.obs['leiden_harmony'].nunique()

        knn_ind = _knn_indices(X_harmony, k=KNN_K)

        # Local batch-mixing metrics
        ilisi_t = ilisi_per_type(X_harmony, batch_labels_sub, ref_labels_sub, knn_ind=knn_ind)
        kbet = kbet_acceptance(X_harmony, batch_labels_sub, knn_ind=knn_ind)
        graph_conn = graph_connectivity_per_type(X_harmony, ref_labels_sub)
        batch_asw_t = batch_asw_per_type(X_harmony, batch_labels_sub, ref_labels_sub)

        # Cross-brain correspondence metrics
        co_purity = co_cluster_purity(post_labels, ref_labels_sub, batch_labels_sub)
        wt_batch_ari = within_type_batch_ari(post_labels, ref_labels_sub, batch_labels_sub)
        cent_dist = centroid_dist_per_type(X_harmony, ref_labels_sub, batch_labels_sub)
        mnn_hit = mnn_hit_rate(X_harmony, batch_labels_sub, ref_labels_sub, knn_ind=knn_ind)
        pair_recovery = hungarian_recovery_rate(
            post_labels, batch_labels_sub, ref_labels_sub, cell_names_sub,
            rna_b1, rna_b2, top_genes, original_types_set, threshold=0.5,
        )

        sigma_results.append({
            'theta': sigma_theta, 'sigma': s, 'nclust': sigma_nclust,
            'sil_batch': sil_batch, 'sil_cluster': sil_cluster,
            'ari': ari, 'nmi': nmi, 'n_clusters': n_clusters,
            'ilisi_per_type': ilisi_t,
            'kbet_accept': kbet,
            'graph_conn_per_type': graph_conn,
            'batch_asw_per_type': batch_asw_t,
            'co_cluster_purity': co_purity,
            'within_type_batch_ari': wt_batch_ari,
            'centroid_dist_per_type': cent_dist,
            'mnn_hit_rate': mnn_hit,
            'pair_recovery_rate': pair_recovery,
            'converged': True,
        })
        print(f"-> silB={sil_batch:.2f} silC={sil_cluster:.2f} "
              f"ARI={ari:.2f} NMI={nmi:.2f} | "
              f"iLISI={ilisi_t:.2f} kBET={kbet:.2f} "
              f"gConn={graph_conn:.2f} bASW={batch_asw_t:.2f} | "
              f"coPur={co_purity:.2f} wtARI={wt_batch_ari:.2f} "
              f"cDist={cent_dist:.2f} MNN={mnn_hit:.2f} "
              f"pRec={pair_recovery:.2f} | k={n_clusters}")

    except Exception as e:
        print(f"-> FAILED: {e}")
        row = {'theta': sigma_theta, 'sigma': s, 'nclust': sigma_nclust, 'converged': False}
        for f in _NAN_FIELDS:
            row[f] = np.nan
        sigma_results.append(row)

sigma_results_df = pd.DataFrame(sigma_results)
sigma_results_df.to_csv("output/harmony_sensitivity_v3_sigma_results.csv", index=False)
print()
print(f"Done. Saved {len(sigma_results_df)} rows to output/harmony_sensitivity_v3_sigma_results.csv")


## 6. Visualize results

In [ ]:
# Metric layout: 3 rows x 5 cols, all thirteen metrics plotted.
# Row 0 = standard global scores (sil_batch, sil_cluster, ARI, NMI),
# Row 1 = local (kNN-based) batch mixing,
# Row 2 = cross-brain correspondence.
SWEEP_METRICS = [
    # (column, title, lower_is_better, row, col)
    # Row 0 -- standard global scores
    ('sil_batch',              'Batch silhouette (|.| lower better)', True,  0, 0),
    ('sil_cluster',            'Cluster silhouette',                  False, 0, 1),
    ('ari',                    'ARI vs matched ref',                  False, 0, 2),
    ('nmi',                    'NMI vs matched ref',                  False, 0, 3),
    # Row 1 -- local / regional batch mixing
    ('graph_conn_per_type',    'Graph connectivity / type',           False, 1, 0),
    ('batch_asw_per_type',     'Batch ASW per type',                  False, 1, 1),
    ('ilisi_per_type',         'iLISI per type',                      False, 1, 2),
    ('kbet_accept',            'kBET acceptance',                     False, 1, 3),
    ('mnn_hit_rate',           'MNN hit rate',                        False, 1, 4),
    # Row 2 -- cross-brain correspondence
    ('co_cluster_purity',      'Co-cluster purity',                   False, 2, 0),
    ('within_type_batch_ari',  'Within-type batch ARI',               True,  2, 1),
    ('centroid_dist_per_type', 'Centroid dist / type',                True,  2, 2),
    ('pair_recovery_rate',     'Hungarian pair recovery',             False, 2, 3),
]

ROW_TITLES = ['Global scores', 'Local batch mixing', 'Cross-brain correspondence']


def _plot_sweep(param, marker, color, title_suffix, out_pdf, df=None, chosen=None):
    if df is None:
        df = results_df
    cols = [m[0] for m in SWEEP_METRICS]
    agg = df.groupby(param)[cols].mean()
    fig, axes = plt.subplots(3, 5, figsize=(24, 12))
    for r in range(3):
        for c in range(5):
            axes[r, c].axis('off')
    for col, title, lower_better, r, c in SWEEP_METRICS:
        ax = axes[r, c]
        ax.axis('on')
        if col == 'sil_batch':
            # Plot |sil_batch| so "closer to 0" reads as "lower" on the y-axis.
            ax.plot(agg.index, agg[col].abs(), marker, linewidth=2.5, color=color)
        else:
            ax.plot(agg.index, agg[col], marker, linewidth=2.5, color=color)
        if chosen is not None:
            ax.axvline(chosen, color='gray', linestyle='--', linewidth=1.8, alpha=0.6)
        ax.set_xlabel(param, fontsize=18)
        ax.tick_params(axis='both', labelsize=16)
        annot = title + (' (lower better)' if lower_better and col != 'sil_batch' else '')
        ax.set_title(annot if annot != title else title, fontsize=18)
        ax.grid(True, alpha=0.3)
    for r, rt in enumerate(ROW_TITLES):
        axes[r, 0].annotate(rt, xy=(-0.30, 0.5), xycoords='axes fraction',
                            rotation=90, ha='center', va='center',
                            fontsize=20)
    plt.suptitle(f'Effect of {param} ({title_suffix})', fontsize=24, y=1.00)
    plt.tight_layout()
    plt.savefig(out_pdf, dpi=200, bbox_inches='tight')
    plt.savefig(out_pdf.rsplit('.', 1)[0] + '.png', dpi=200, bbox_inches='tight')
    plt.show()


### 6a. Theta sweep

In [ ]:
_plot_sweep('theta', 'o-', 'tab:blue',
            f'sigma={sigma_values[0]}, averaged over nclust',
            'v3_theta_sweep.pdf', chosen=6)


### 6b. nclust sweep

In [ ]:
_plot_sweep('nclust', 'D-', 'tab:green',
            f'sigma={sigma_values[0]}, averaged over theta',
            'v3_nclust_sweep.pdf', chosen=100)


### 6c. Sigma sweep (theta=6, nclust=100)

In [ ]:
_plot_sweep('sigma', '^-', 'tab:purple',
            f'theta={sigma_theta}, nclust={sigma_nclust}',
            'v3_sigma_sweep.pdf', df=sigma_results_df, chosen=0.02)


### 6d. Trade-off plots

In [ ]:
# Original trade-off: batch silhouette vs cluster silhouette, colored by theta.
fig, ax = plt.subplots(figsize=(8, 6))
sc_plot = ax.scatter(
    results_df['sil_batch'], results_df['sil_cluster'],
    c=results_df['theta'], cmap='coolwarm',
    s=40, alpha=0.7, edgecolors='k', linewidths=0.3
)
ax.axvline(0, color='gray', linestyle='--', linewidth=0.5)
ax.set_xlabel('Batch silhouette (closer to 0 = better mixing)')
ax.set_ylabel('Cluster silhouette (higher = better separation)')
ax.set_title('Trade-off: batch correction vs cluster quality')
cbar = plt.colorbar(sc_plot, ax=ax); cbar.set_label('theta')
plt.tight_layout()
plt.savefig('output/v3_tradeoff.pdf', dpi=200, bbox_inches='tight')
plt.show()


In [ ]:
# New trade-off using the local/correspondence metrics:
# x = local batch mixing (iLISI per type), y = cross-brain co-clustering purity.
# Top-right corner is what we want; old `sil_batch ~ 0` runs may sit far from it.
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

ax = axes[0]
sc_plot = ax.scatter(
    results_df['ilisi_per_type'], results_df['co_cluster_purity'],
    c=results_df['theta'], cmap='coolwarm',
    s=40, alpha=0.8, edgecolors='k', linewidths=0.3
)
ax.set_xlabel('iLISI per type (higher = better local mixing)')
ax.set_ylabel('Co-cluster purity (higher = matched types co-cluster)')
ax.set_title('Mixing vs cross-brain correspondence')
cbar = plt.colorbar(sc_plot, ax=ax); cbar.set_label('theta')

ax = axes[1]
sc_plot = ax.scatter(
    results_df['within_type_batch_ari'], results_df['centroid_dist_per_type'],
    c=results_df['theta'], cmap='coolwarm',
    s=40, alpha=0.8, edgecolors='k', linewidths=0.3
)
ax.set_xlabel('Within-type batch ARI (lower = better co-clustering)')
ax.set_ylabel('Centroid dist / type (lower = better alignment)')
ax.set_title('Within-type residual batch effect')
cbar = plt.colorbar(sc_plot, ax=ax); cbar.set_label('theta')

plt.tight_layout()
plt.savefig('output/v3_tradeoff_correspondence.pdf', dpi=200, bbox_inches='tight')
plt.show()


### 6e. Heatmaps: theta vs nclust interaction

In [ ]:
HEATMAP_METRICS = [
    ('sil_batch',              '|Batch silhouette| (lower better)',    'YlOrRd_r', True),
    ('sil_cluster',            'Cluster silhouette',                   'YlOrRd',   False),
    ('ari',                    'ARI vs matched ref',                   'YlOrRd',   False),
    ('nmi',                    'NMI vs matched ref',                   'YlOrRd',   False),
    ('graph_conn_per_type',    'Graph connectivity / type',            'YlOrRd',   False),
    ('batch_asw_per_type',     'Batch ASW per type',                   'YlOrRd',   False),
    ('ilisi_per_type',         'iLISI per type',                       'YlOrRd',   False),
    ('kbet_accept',            'kBET acceptance',                      'YlOrRd',   False),
    ('mnn_hit_rate',           'MNN hit rate',                         'YlOrRd',   False),
    ('co_cluster_purity',      'Co-cluster purity',                    'YlOrRd',   False),
    ('within_type_batch_ari',  'Within-type batch ARI (lower better)', 'YlOrRd_r', False),
    ('centroid_dist_per_type', 'Centroid dist / type (lower better)',  'YlOrRd_r', False),
    ('pair_recovery_rate',     'Hungarian pair recovery',              'YlOrRd',   False),
]

for metric, title, cmap, take_abs in HEATMAP_METRICS:
    series = results_df[metric].abs() if take_abs else results_df[metric]
    pivot_df = results_df.assign(_v=series).pivot_table(
        index='theta', columns='nclust', values='_v', aggfunc='mean')
    fig, ax = plt.subplots(figsize=(7, 4.2))
    sns.heatmap(pivot_df, annot=True, fmt='.3f', cmap=cmap, ax=ax, linewidths=0)
    ax.set_title(f'{title} (theta x nclust, sigma={sigma_values[0]})', fontsize=12)
    plt.tight_layout()
    plt.show()


### 6f. Combined sweep figure

Single figure stacking all three sweeps (`theta`, `nclust`, `sigma`) on the same
13-metric panel. Each sweep section is a 3-row x 5-col grid grouped by metric
category; the chosen parameter value (`theta=6, nclust=100, sigma=0.02`) is
marked with a grey dashed line on every panel. 

In [ ]:
# Combined sweep figure: 9 rows x 5 cols (3 sweep sections, each a 3x5 metric grid).
def _plot_three_sweeps_combined(out_pdf):
    sweep_specs = [
        ('theta',  results_df,        6,    'o-', 'tab:blue',
         f'theta sweep (sigma={sigma_values[0]}, averaged over nclust)'),
        ('nclust', results_df,        100,  'D-', 'tab:green',
         f'nclust sweep (sigma={sigma_values[0]}, averaged over theta)'),
        ('sigma',  sigma_results_df,  0.02, '^-', 'tab:purple',
         f'sigma sweep (theta={sigma_theta}, nclust={sigma_nclust})'),
    ]

    n_sweeps = len(sweep_specs)
    n_metric_rows = 3
    n_metric_cols = 5
    cols = [m[0] for m in SWEEP_METRICS]

    fig, axes = plt.subplots(n_sweeps * n_metric_rows, n_metric_cols,
                              figsize=(24, 4.5 * n_sweeps * n_metric_rows))

    # Hide all panels first; we re-enable only those backed by a metric.
    for ax in axes.ravel():
        ax.axis('off')

    for sweep_idx, (param, df, chosen, marker, color, sweep_title) in enumerate(sweep_specs):
        agg = df.groupby(param)[cols].mean()
        row_offset = sweep_idx * n_metric_rows

        for col, mtitle, lower_better, r, c in SWEEP_METRICS:
            ax = axes[row_offset + r, c]
            ax.axis('on')
            if col == 'sil_batch':
                ax.plot(agg.index, agg[col].abs(), marker, linewidth=2.5, color=color)
            else:
                ax.plot(agg.index, agg[col], marker, linewidth=2.5, color=color)
            ax.axvline(chosen, color='gray', linestyle='--', linewidth=1.8, alpha=0.6)
            ax.set_xlabel(param, fontsize=16)
            ax.tick_params(axis='both', labelsize=14)
            annot = mtitle + (' (lower better)' if lower_better and col != 'sil_batch' else '')
            ax.set_title(annot, fontsize=16)
            ax.grid(True, alpha=0.3)

        # Sweep-section header (shown above the leftmost column of each section).
        # Use the first panel of the sweep to host both the param label and the
        # row-1 metric category label without overlapping.
        axes[row_offset, 0].annotate(sweep_title,
                                      xy=(-0.55, 1.30), xycoords='axes fraction',
                                      ha='left', va='center',
                                      fontsize=24, fontweight='bold')

        # Metric-category labels (Global / Local mixing / Correspondence) for
        # this sweep section.
        for r, rt in enumerate(ROW_TITLES):
            axes[row_offset + r, 0].annotate(rt, xy=(-0.30, 0.5), xycoords='axes fraction',
                                              rotation=90, ha='center', va='center',
                                              fontsize=12)

    plt.suptitle(
        'Harmony parameter sensitivity: combined theta / nclust / sigma sweeps\n'
        '(chosen values: theta=6, sigma=0.02, nclust=100; dashed lines)',
        fontsize=26, y=1.001,
    )
    plt.tight_layout()
    plt.savefig(out_pdf, dpi=200, bbox_inches='tight')
    plt.savefig(out_pdf.rsplit('.', 1)[0] + '.png', dpi=200, bbox_inches='tight')
    plt.show()


_plot_three_sweeps_combined('v3_combined_sweep.pdf')
